# Statistical testing: is composition actually different from the alternative?

This notebook runs a paired Wilcoxon signed-rank test on the per-sentence NLL differences for
a focused set of comparisons tied directly to the two settings' research questions (not all
pairwise combinations, most of which wouldn't answer anything anyone asked).

## 0. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

RESULTS_DIR = Path("results")
ALPHA = 0.05

zero_shot = np.load(RESULTS_DIR / "zero_shot_per_example_losses.npz")
supervised = np.load(RESULTS_DIR / "supervised_per_example_losses.npz")

# one flat dict of condition_name > per-sentence NLL array, pulled from both files
losses = {name: zero_shot[name] for name in zero_shot.files}
losses.update({name: supervised[name] for name in supervised.files})

for name, arr in losses.items():
    print(f"{name:<18} n={len(arr)}")

# sanity check: every array must be the same length and (by construction) the same sentence order
lengths = {len(a) for a in losses.values()}
assert len(lengths) == 1, f"array length mismatch: {lengths}"
print(f"\nall {len(losses)} conditions have {lengths.pop()} paired per-sentence losses -- OK")

backbone           n=2618
genre_only         n=2618
language_only      n=2618
composed           n=2618
composed_adapted   n=2618
backbone_adapted   n=2618

all 6 conditions have 2618 paired per-sentence losses -- OK


## 1. Paired-test helper

For each comparison, `condition_a` is the "test" condition (the one whose name is asked about
first) and `condition_b` is what it's compared against. Reported per comparison:

- **p-value** (two-sided Wilcoxon signed-rank, on the per-sentence differences `a - b`)
- **median difference** (in NLL) -- the practical size of the effect, sign-consistent with `a - b`
  (negative = `a` typically *lower* loss, i.e. *better*, than `b`)
- **win rate** -- the fraction of individual sentences where `a` actually has the lower (better)
  loss than `b`. This is the most intuitive check on whether an effect is a consistent
  sentence-by-sentence pattern or a few large outliers dragging a mean/median around.

In [ ]:
def paired_test(name_a, name_b):
    a, b = losses[name_a], losses[name_b]
    diff = a - b  # negative -> a better (lower loss) on that sentence


    stat, p = wilcoxon(diff, alternative="two-sided")

    return {
        "comparison": f"{name_a} vs {name_b}",
        "median_diff_nll": float(np.median(diff)),
        "win_rate_a": float((diff < 0).mean()),   # fraction of sentences where a < b (a wins)
        "tie_rate": float((diff == 0).mean()),
        "n": len(diff),
        "statistic": float(stat),
        "p_value": float(p),
    }

## 2. Comparisons

Tied directly to the two settings' actual questions (see `report_statistical_testing.md` for the
full rationale for each):

**Zero-shot (Setting 1)** -- does composing help or hurt, relative to no adaptation and relative to
the better single module?
- `composed` vs `backbone`
- `composed` vs `genre_only`
- `composed` vs `language_only`

**Supervised (Setting 2)** -- the central question: does pre-composing the modules help over
adapting from scratch on the same data?
- `composed_adapted` vs `backbone_adapted`

**Sanity checks** -- did the adaptation step itself help each starting point? (expected yes, given
the large perplexity gap already visible; confirming formally rather than assuming)
- `composed_adapted` vs `composed`
- `backbone_adapted` vs `backbone`

In [3]:
COMPARISONS = [
    ("composed", "backbone"),
    ("composed", "genre_only"),
    ("composed", "language_only"),
    ("composed_adapted", "backbone_adapted"),
    ("composed_adapted", "composed"),
    ("backbone_adapted", "backbone"),
]

results = pd.DataFrame([paired_test(a, b) for a, b in COMPARISONS])
results

,comparison,median_diff_nll,win_rate_a,tie_rate,n,statistic,p_value
0,composed vs backbone,0.330225,0.106570,0.0,2618,199840.5,0.000000e+00
1,composed vs genre_only,0.376520,0.044691,0.0,2618,84779.5,0.000000e+00
2,composed vs language_only,0.113555,0.273873,0.0,2618,852335.0,5.753877e-110
3,composed_adapted vs backbone_adapted,0.007995,0.422842,0.0,2618,1349829.5,4.579234e-21
4,composed_adapted vs composed,-0.701399,0.997708,0.0,2618,74.0,0.000000e+00
5,backbone_adapted vs backbone,-0.390159,0.988159,0.0,2618,14694.5,0.000000e+00


## 3. Holm-Bonferroni correction

Six tests are run together here, not one -- reporting six raw p-values as if each were the only
test performed would understate the overall false-positive risk. Holm-Bonferroni controls the
family-wise error rate while being less conservative than a flat Bonferroni correction: p-values
are sorted ascending, and each is compared against a progressively looser threshold
(`alpha / (m - k + 1)` for the k-th smallest of m p-values), stepping down until one fails to
clear its threshold.

In [4]:
def holm_bonferroni(p_values, alpha=ALPHA):
    m = len(p_values)
    order = np.argsort(p_values)
    adjusted = np.empty(m)
    running_max = 0.0
    for rank, idx in enumerate(order):  # rank is 0-indexed here, so use (m - rank) below
        adj = (m - rank) * p_values[idx]
        running_max = max(running_max, adj)
        adjusted[idx] = min(running_max, 1.0)  # enforce monotonicity + cap at 1
    return adjusted

results["p_value_holm"] = holm_bonferroni(results["p_value"].to_numpy())
results["significant_holm"] = results["p_value_holm"] < ALPHA
results

,comparison,median_diff_nll,win_rate_a,tie_rate,n,statistic,p_value,p_value_holm,significant_holm
0,composed vs backbone,0.330225,0.106570,0.0,2618,199840.5,0.000000e+00,0.000000e+00,True
1,composed vs genre_only,0.376520,0.044691,0.0,2618,84779.5,0.000000e+00,0.000000e+00,True
2,composed vs language_only,0.113555,0.273873,0.0,2618,852335.0,5.753877e-110,1.150775e-109,True
3,composed_adapted vs backbone_adapted,0.007995,0.422842,0.0,2618,1349829.5,4.579234e-21,4.579234e-21,True
4,composed_adapted vs composed,-0.701399,0.997708,0.0,2618,74.0,0.000000e+00,0.000000e+00,True
5,backbone_adapted vs backbone,-0.390159,0.988159,0.0,2618,14694.5,0.000000e+00,0.000000e+00,True


## 4. Readable summary

In [5]:
for _, row in results.iterrows():
    direction = "lower (better)" if row["median_diff_nll"] < 0 else "higher (worse)"
    sig = "significant" if row["significant_holm"] else "not significant"
    print(
        f"{row['comparison']:<35} median NLL diff = {row['median_diff_nll']:+.4f} "
        f"({direction}), win_rate={row['win_rate_a']:.1%}, "
        f"p={row['p_value']:.2e}, p_holm={row['p_value_holm']:.2e} -> {sig}"
    )

composed vs backbone                median NLL diff = +0.3302 (higher (worse)), win_rate=10.7%, p=0.00e+00, p_holm=0.00e+00 -> significant
composed vs genre_only              median NLL diff = +0.3765 (higher (worse)), win_rate=4.5%, p=0.00e+00, p_holm=0.00e+00 -> significant
composed vs language_only           median NLL diff = +0.1136 (higher (worse)), win_rate=27.4%, p=5.75e-110, p_holm=1.15e-109 -> significant
composed_adapted vs backbone_adapted median NLL diff = +0.0080 (higher (worse)), win_rate=42.3%, p=4.58e-21, p_holm=4.58e-21 -> significant
composed_adapted vs composed        median NLL diff = -0.7014 (lower (better)), win_rate=99.8%, p=0.00e+00, p_holm=0.00e+00 -> significant
backbone_adapted vs backbone        median NLL diff = -0.3902 (lower (better)), win_rate=98.8%, p=0.00e+00, p_holm=0.00e+00 -> significant


In [6]:
results.to_csv(RESULTS_DIR / "statistical_tests.csv", index=False)
print("saved results/statistical_tests.csv")

saved results/statistical_tests.csv
